In [1]:
%pip install -q xgboost
%pip install -q s3fs

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load in our libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "vscode"

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


In [3]:
data = pd.read_csv('https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/stacking/titanic.csv')

data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
display(data.describe(include='all'))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [6]:
# Age, Cabin and Embarked have missing values.
# Quantitative variables needs to be normalized.
# Qualitative variables needs to be encoded.

# Can run median or nearest nezighbors preproccessing for Age?  keep null values for Embarked and Cabin cuz it's maybe a valuable info.


In [7]:
# Creating a Name_lenght variable to measure the number of characters in Names
data['Name_lenght'] = data['Name'].apply(len)

# Creating a variable Has_Cabin to indicate if the passenger has a cabin or not
data['Has_cabin'] = data['Cabin'].notna()


# Creating FamilySize as combination of SibSp and Parch: size of each family
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1 # Person himself ?

# Creating a variable IsAlone if the passenger is traveling on thei own
data['IsAlone'] = data['FamilySize'] == 1

# Extracting the title of each Name to create a variable Title
def get_title(name):
    return name.split(", ")[1].split(".")[0]

data['Title'] = data['Name'].apply(get_title)

data.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_lenght,Has_cabin,FamilySize,IsAlone,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False,2,False,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True,2,False,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False,1,True,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True,2,False,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False,1,True,Mr


In [8]:
px.bar(data, 'Title')


In [9]:
data['Title'] = data['Title'].replace('Mlle', 'Miss')
data['Title'] = data['Title'].replace('Ms', 'Miss')
data['Title'] = data['Title'].replace('Mme', 'Mrs')

px.bar(data, 'Title')

In [10]:
# Grouping all non_common titles together into a a "Rare" modality
rare_titles = ['Don', 'Mme', 'Ms', 'Major', 'Lady', 'Sir', 'Mlle', 'Col', 'Capt', 'the Countess', 'Jonkheer']
data['Title'] = data['Title'].replace(rare_titles, 'Rare')

px.histogram(data, 'Title')

In [11]:
# Feature selection

drop_elements = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp']
data = data.drop(drop_elements, axis=1)

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Survived     891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Sex          891 non-null    str    
 3   Age          714 non-null    float64
 4   Parch        891 non-null    int64  
 5   Fare         891 non-null    float64
 6   Embarked     889 non-null    str    
 7   Name_lenght  891 non-null    int64  
 8   Has_cabin    891 non-null    bool   
 9   FamilySize   891 non-null    int64  
 10  IsAlone      891 non-null    bool   
 11  Title        891 non-null    str    
dtypes: bool(2), float64(2), int64(5), str(3)
memory usage: 71.5 KB


In [12]:
# Separate X and Y, split test and train

X = data.drop('Survived', axis=1)

Y = data['Survived']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, stratify= Y, random_state=0) # test_size default is 0.25

X.dtypes



Pclass           int64
Sex                str
Age            float64
Parch            int64
Fare           float64
Embarked           str
Name_lenght      int64
Has_cabin         bool
FamilySize       int64
IsAlone           bool
Title              str
dtype: object

In [13]:
numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number', 'bool']).columns.tolist()

print(f'Numerical features are: {numeric_features}')
print(f'Categorical features are: {categorical_features}')

Numerical features are: ['Pclass', 'Age', 'Parch', 'Fare', 'Name_lenght', 'Has_cabin', 'FamilySize', 'IsAlone']
Categorical features are: ['Sex', 'Embarked', 'Title']


In [14]:
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer()),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

X_train = preprocessor.fit_transform(X_train)
print(X_train[0:5,:])
print()

X_test = preprocessor.transform(X_test)
print(X_test[0:5,:])
print()


[[-1.54562952  0.07998138 -0.46398808  0.35047578 -0.52725961  1.83466974
   0.08547193 -1.25268941  1.          0.          1.          0.
   0.          1.          0.          0.          0.        ]
 [-0.35613584 -0.89717426  2.08699167 -0.37942193  0.23157878 -0.54505723
   0.72699155 -1.25268941  0.          0.          1.          0.
   1.          0.          0.          0.          0.        ]
 [ 0.83335785 -1.57366663  2.08699167 -0.10056358 -0.96088154 -0.54505723
   2.65155041 -1.25268941  0.          0.          1.          0.
   1.          0.          0.          0.          0.        ]
 [-0.35613584 -0.59651099  0.81150179 -0.07997672  2.61649942 -0.54505723
   0.72699155 -1.25268941  0.          0.          1.          0.
   0.          0.          1.          0.          0.        ]
 [ 0.83335785 -0.82200845 -0.46398808 -0.47720951  0.01476781 -0.54505723
  -0.5560477   0.79828247  1.          0.          1.          0.
   0.          1.          0.          0.       

In [15]:
# Correlation matrix
column_names = [
    nom.removeprefix("num__").removeprefix("cat__")
    for nom in preprocessor.get_feature_names_out()
]

X_train_df = pd.DataFrame(X_train, columns=column_names)
corr_matrix = X_train_df.corr().round(2)


import plotly.figure_factory as ff

fig = ff.create_annotated_heatmap(corr_matrix.values,
                                  x = corr_matrix.columns.tolist(),
                                  y = corr_matrix.index.tolist())


fig.show()


In [16]:
scores_df = pd.DataFrame(columns = ['model', 'accuracy', 'set'])

In [17]:
random_forest = RandomForestClassifier()

params = {
    'max_depth': [2, 4, 6, 8, 10],
    'min_samples_leaf': [1, 2, 5],
    'min_samples_split': [2, 4, 8],
    'n_estimators': [10, 20, 40, 60, 80, 100]
}


gridsearch = GridSearchCV(random_forest, param_grid=params, cv=3, verbose=1)
gridsearch.fit(X_train, Y_train)
print()
print(f'Best hyperparamters are: {gridsearch.best_params_}')
print()
print(f'Best valdiation accuracy: {gridsearch.best_score_}')
print()
print(f'Accuracy on training set: {gridsearch.score(X_train, Y_train)}')
print(f'Accuracy on test set: {gridsearch.score(X_test, Y_test)}')

Fitting 3 folds for each of 270 candidates, totalling 810 fits

Best hyperparamters are: {'max_depth': 8, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 40}

Best valdiation accuracy: 0.8263375483106424

Accuracy on training set: 0.9281437125748503
Accuracy on test set: 0.8026905829596412


In [18]:
new_rows = [
    {'model': 'random_forest', 'accuracy': gridsearch.score(X_train, Y_train), 'set': 'train'},
    {'model': 'random_forest', 'accuracy': gridsearch.score(X_test, Y_test), 'set': 'test'}
]

scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df

,model,accuracy,set
0,random_forest,0.928144,train
1,random_forest,0.802691,test


In [21]:
# Bagging of decision tree

decision_tree = DecisionTreeClassifier(max_depth=8, min_samples_leaf=1, min_samples_split=4)
bagging = BaggingClassifier(estimator=decision_tree, n_estimators=40)
bagging.fit(X_train, Y_train)

print("Accuracy on training set : ", bagging.score(X_train, Y_train))
print("Accuracy on test set : ", bagging.score(X_test, Y_test))

new_rows = [
    {'model': 'bagging_dt', 'accuracy': bagging.score(X_train, Y_train), 'set': 'train'},
    {'model': 'bagging_dt', 'accuracy': bagging.score(X_test, Y_test), 'set': 'test'}
]

scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df

Accuracy on training set :  0.9416167664670658
Accuracy on test set :  0.8251121076233184


,model,accuracy,set
0,random_forest,0.928144,train
1,random_forest,0.802691,test
2,bagging_dt,0.941617,train
3,bagging_dt,0.825112,test


In [ ]:
# Gradien Boost

gradientboost = GradientBoostingClassifier()

# Grid of values to be tested
params = {
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [6, 8, 10],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}
print(params)
gridsearch = GridSearchCV(gradientboost, param_grid = params, cv = 3, verbose = 1)
gridsearch.fit(X_train, Y_train)

print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

new_rows = [
    {'model': 'gradient_boost', 'accuracy': gridsearch.score(X_train, Y_train), 'set': 'train'},
    {'model': 'gradient_boost', 'accuracy': gridsearch.score(X_test, Y_test), 'set': 'test'}
]

scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df


{'max_depth': [8, 10, 12], 'min_samples_leaf': [1, 2, 3], 'min_samples_split': [6, 8, 10], 'n_estimators': [2, 4, 6, 8, 10, 12]}
Fitting 3 folds for each of 162 candidates, totalling 486 fits
Best hyperparameters :  {'max_depth': 8, 'min_samples_leaf': 3, 'min_samples_split': 10, 'n_estimators': 10}
Best validation accuracy :  0.8158539705624909

Accuracy on training set :  0.9326347305389222
Accuracy on test set :  0.820627802690583


,model,accuracy,set
0,random_forest,0.928144,train
1,random_forest,0.802691,test
2,bagging_dt,0.941617,train
3,bagging_dt,0.825112,test
4,gradient_boost,0.932635,train
5,gradient_boost,0.820628,test


In [24]:
#XGBoost

xgboost = XGBClassifier()

# Grid of values to be tested
params = {
    'max_depth': [4, 6, 8, 10],
    'min_child_weight': [1, 2, 4, 6, 8],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}
print(params)
gridsearch = GridSearchCV(xgboost, param_grid = params, cv = 3, verbose = 1) 
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

new_rows = [
    {'model': 'xgboost', 'accuracy': gridsearch.score(X_train, Y_train), 'set': 'train'},
    {'model': 'xgboost', 'accuracy': gridsearch.score(X_test, Y_test), 'set': 'test'}
]

scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df


{'max_depth': [4, 6, 8, 10], 'min_child_weight': [1, 2, 4, 6, 8], 'n_estimators': [2, 4, 6, 8, 10, 12]}
Fitting 3 folds for each of 120 candidates, totalling 360 fits
...Done.
Best hyperparameters :  {'max_depth': 4, 'min_child_weight': 4, 'n_estimators': 8}
Best validation accuracy :  0.8308016536716089

Accuracy on training set :  0.8622754491017964
Accuracy on test set :  0.8026905829596412


,model,accuracy,set
0,random_forest,0.928144,train
1,random_forest,0.802691,test
2,bagging_dt,0.941617,train
3,bagging_dt,0.825112,test
4,gradient_boost,0.932635,train
5,gradient_boost,0.820628,test
6,xgboost,0.862275,train
7,xgboost,0.802691,test


In [25]:
scores_df = scores_df.sort_values(by = ['set', 'accuracy'], ascending = False)
scores_df


,model,accuracy,set
2,bagging_dt,0.941617,train
4,gradient_boost,0.932635,train
0,random_forest,0.928144,train
6,xgboost,0.862275,train
3,bagging_dt,0.825112,test
5,gradient_boost,0.820628,test
1,random_forest,0.802691,test
7,xgboost,0.802691,test


In [26]:
scores_df = scores_df.sort_values(by = ['set', 'accuracy'], ascending = False)
scores_df


,model,accuracy,set
2,bagging_dt,0.941617,train
4,gradient_boost,0.932635,train
0,random_forest,0.928144,train
6,xgboost,0.862275,train
3,bagging_dt,0.825112,test
5,gradient_boost,0.820628,test
1,random_forest,0.802691,test
7,xgboost,0.802691,test


In [29]:
px.bar(scores_df, x = "model", y = 'accuracy', color = 'set', barmode = 'group', width = 900)


In [28]:
print('The gradient boosting model seems to be a bit better than the other ones: it has the best test accuracy and still doesnt overfit that much.')

The gradient boosting model seems to be a bit better than the other ones: it has the best test accuracy and still doesnt overfit that much.
